# Lab 10: MapReduce Word Count

This notebook implements Word Count using Hadoop Streaming (mapper and reducer in Python).
It also runs a local CPU baseline and compares execution time with the Hadoop job.

In [1]:
%%writefile wc_input.txt
Hadoop makes large-scale data processing easier.
Word count is the classic MapReduce example.
Hadoop and Python streaming are useful for data engineering.
Word count, word count, and more word count.

Writing wc_input.txt


In [2]:
%%writefile mapper_wc.py
import re
import sys

for line in sys.stdin:
    for word in re.findall(r"[A-Za-z0-9']+", line.lower()):
        print(f"{word}\t1")

Writing mapper_wc.py


In [3]:
%%writefile reducer_wc.py
import sys

current_word = None
running_total = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    word, count = line.split("\t", 1)
    count = int(count)

    if current_word is None:
        current_word = word
        running_total = count
    elif word == current_word:
        running_total += count
    else:
        print(f"{current_word}\t{running_total}")
        current_word = word
        running_total = count

if current_word is not None:
    print(f"{current_word}\t{running_total}")

Writing reducer_wc.py


In [4]:
%%bash
set -e
cat wc_input.txt | python3 mapper_wc.py | sort | python3 reducer_wc.py > wc_local_output.txt
echo "Local mapper+reducer output:"
cat wc_local_output.txt

Local mapper+reducer output:
and	2
are	1
classic	1
count	4
data	2
easier	1
engineering	1
example	1
for	1
hadoop	2
is	1
large	1
makes	1
mapreduce	1
more	1
processing	1
python	1
scale	1
streaming	1
the	1
useful	1
word	4


In [5]:
import re
import time
from collections import Counter

start = time.perf_counter()
counts = Counter()
with open("wc_input.txt", "r", encoding="utf-8") as f:
    for line in f:
        for word in re.findall(r"[A-Za-z0-9']+", line.lower()):
            counts[word] += 1
elapsed_ms = (time.perf_counter() - start) * 1000.0

with open("wc_cpu_output.txt", "w", encoding="utf-8") as out:
    for word in sorted(counts):
        out.write(f"{word}\t{counts[word]}\n")

with open("wc_cpu_time_ms.txt", "w", encoding="utf-8") as t:
    t.write(f"{elapsed_ms:.6f}")

print(f"CPU baseline completed in {elapsed_ms:.3f} ms")

CPU baseline completed in 0.706 ms


In [6]:
%%bash
set -e

if ! command -v java >/dev/null 2>&1; then
  apt-get update -qq
  apt-get install -y -qq openjdk-11-jdk-headless
fi

JAVA_BIN="$(readlink -f "$(command -v java)")"
JAVA_HOME_DIR="$(dirname "$(dirname "$JAVA_BIN")")"

if command -v hadoop >/dev/null 2>&1; then
  HADOOP_BIN="$(command -v hadoop)"
  HADOOP_HOME="$(cd "$(dirname "$HADOOP_BIN")/.." && pwd)"
else
  HADOOP_VERSION=3.3.6
  if [ ! -d "$HOME/hadoop-$HADOOP_VERSION" ]; then
    wget -q "https://archive.apache.org/dist/hadoop/common/hadoop-$HADOOP_VERSION/hadoop-$HADOOP_VERSION.tar.gz"
    tar -xzf "hadoop-$HADOOP_VERSION.tar.gz" -C "$HOME"
  fi
  HADOOP_HOME="$HOME/hadoop-$HADOOP_VERSION"
fi

echo "export JAVA_HOME=$JAVA_HOME_DIR" > hadoop_env.sh
echo "export HADOOP_HOME=$HADOOP_HOME" >> hadoop_env.sh
echo 'export PATH=$HADOOP_HOME/bin:$PATH' >> hadoop_env.sh

source hadoop_env.sh
hadoop version | head -n 1

Hadoop 3.3.6


In [7]:
%%bash
set -e

if [ -f hadoop_env.sh ]; then
  source hadoop_env.sh
fi

if ! command -v hadoop >/dev/null 2>&1; then
  echo "Hadoop not found. Run the Hadoop setup cell first." >&2
  exit 1
fi

STREAMING_JAR=$(ls "$HADOOP_HOME"/share/hadoop/tools/lib/hadoop-streaming*.jar 2>/dev/null | head -n 1)
if [ -z "$STREAMING_JAR" ]; then
  echo "Could not find hadoop-streaming jar under $HADOOP_HOME/share/hadoop/tools/lib" >&2
  exit 1
fi

rm -rf output_wc
LOG_FILE=wc_hadoop.log
START_MS=$(date +%s%3N)
set +e
hadoop jar "$STREAMING_JAR" \
  -D mapreduce.framework.name=local \
  -files mapper_wc.py,reducer_wc.py \
  -input wc_input.txt \
  -output output_wc \
  -mapper "python3 mapper_wc.py" \
  -reducer "python3 reducer_wc.py" > "$LOG_FILE" 2>&1
STATUS=$?
set -e
END_MS=$(date +%s%3N)

if [ $STATUS -ne 0 ]; then
  echo "Hadoop word count job failed. Last 80 log lines:" >&2
  tail -n 80 "$LOG_FILE" >&2
  exit $STATUS
fi

echo $((END_MS - START_MS)) > wc_hadoop_time_ms.txt
cat output_wc/part-* | sort > wc_hadoop_output.txt
echo "Hadoop output:"
cat wc_hadoop_output.txt

Hadoop output:
and	2
are	1
classic	1
count	4
data	2
easier	1
engineering	1
example	1
for	1
hadoop	2
is	1
large	1
makes	1
mapreduce	1
more	1
processing	1
python	1
scale	1
streaming	1
the	1
useful	1
word	4


In [8]:
from pathlib import Path

cpu_time_ms = float(Path("wc_cpu_time_ms.txt").read_text(encoding="utf-8").strip())
hadoop_time_ms = float(Path("wc_hadoop_time_ms.txt").read_text(encoding="utf-8").strip())

cpu_output = Path("wc_cpu_output.txt").read_text(encoding="utf-8").strip().splitlines()
hadoop_output = Path("wc_hadoop_output.txt").read_text(encoding="utf-8").strip().splitlines()

print(f"Output match: {cpu_output == hadoop_output}")
print(f"CPU baseline time: {cpu_time_ms:.3f} ms")
print(f"Hadoop streaming time: {hadoop_time_ms:.3f} ms")
if cpu_time_ms > 0:
    print(f"Relative overhead (Hadoop/CPU): {hadoop_time_ms / cpu_time_ms:.2f}x")
print("GPU timing is not applicable for this Hadoop Streaming lab.")

Output match: True
CPU baseline time: 0.706 ms
Hadoop streaming time: 5467.000 ms
Relative overhead (Hadoop/CPU): 7740.62x
GPU timing is not applicable for this Hadoop Streaming lab.
